# Инференс

Модель **49M / 300k** и режим **Top-k** (`k=40`, `temperature=0.8`), выбранные по результатам сравнения.

## 1. Загрузка модели

In [1]:
from pathlib import Path
import json
import sys

import torch

project_root = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src/ggpt/inference/artifact.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Откройте ноутбук из директории проекта.")

src = str(project_root / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from ggpt.inference.artifact import export_model, load_model_artifact
from ggpt.inference.generate import GenerationConfig
from ggpt.inference.model_runner import GGPTTextRunner

model_dir = project_root / "artifacts/exports/llama_49m_300k"
run_dir = project_root / "runs/ggpt/llama_49m_300k_v1"
device = "cuda" if torch.cuda.is_available() else "cpu"

if not model_dir.exists():
    if not (run_dir / "checkpoints/latest.pt").is_file():
        raise FileNotFoundError(
            "Не найден экспорт модели и checkpoint 300k. "
            f"Ожидался экспорт: {model_dir}"
        )
    print("Экспорт inference-модели из checkpoint 300k...")
    export_model(run_dir, model_dir, checkpoint="latest")

manifest = json.loads((model_dir / "manifest.json").read_text(encoding="utf-8"))
if manifest["step"] != 300_000:
    raise ValueError(f"Ожидался checkpoint 300000, найден {manifest['step']}")

model, tokenizer, manifest = load_model_artifact(model_dir, device=device)
runner = GGPTTextRunner(model, tokenizer, device)

print(f"Модель: GGPT 49M / {manifest['step']:,} шагов")
print(f"Устройство: {device}")
print(f"Контекст: {model.config.max_seq_len} токенов")


Модель: GGPT 49M / 300,000 шагов
Устройство: cuda
Контекст: 512 токенов


## 2. Настройки генерации

Настройки соответствуют режиму `topk_t08` из Benchmark v3.

In [2]:
generation_settings = GenerationConfig(
    max_new_tokens=96,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    seed=42,
)


## 3. Генерация

In [3]:
prompt = "ერთხელ პატარა სოფელში ცხოვრობდა"

continuation = runner.generate(prompt, generation_settings)

print("Промпт:")
print(prompt)
print("\nПродолжение:")
print(continuation)


Промпт:
ერთხელ პატარა სოფელში ცხოვრობდა

Продолжение:
. როცა პატარა ბიჭი დაეღუპა, დედა და მამა არ ეძებდნენ, მაგრამ მაინც იმ იმედით დადიოდნენ, რომ ყველაფერი თავიდან დაიწყებოდა და მალე ამ ყველაფრის შემდეგ მათ ცხოვრებაში ახალი ეტაპი დაიწყება.
ამ დროს მშობლები გაიგებენ – მამა ცოცხალი იყო, დედა – ცოცხალი, დედა – ცოცხალი.
ამ დროს დედა, რომელიც მეორე სახლში წავიდა, აღარ იყო.
როდესაც მშობლები სახლში დაბრუნდნენ, დედა და მამა ისევ სახლში დარჩნენ. მამა სახლში დარჩა, შვილმა კი მშობლების სახლში
